# Perfilamiento de Datos

Presentado por:

*  Isabela Cáceres
* Juan Pablo Mosquera
* Julian Plata

En el presente documento se encuentra el perfilamiento de datos de las fuentes de datos:

* **classicmodels:** la cual esta contenida en el sql: `classicmodels_mysql.sql`

* **customerservice:** la cual esta en el archivo: `customerservice.sql`

### Instalación de librería

In [ ]:
!pip install pandas openpyxl  pysqlite3-binary mysql-connector-python

## Imports

In [ ]:
import mysql.connector
import psycopg2
import pandas as pd
import sqlite3
import re
import numpy as np

## Carga bases de datos

In [ ]:

def limpiar_puntoycoma_en_strings(sql):
    resultado = []
    en_string = False

    for c in sql:
        if c == "'":
            en_string = not en_string
            resultado.append(c)
        elif c == ";" and en_string:
            resultado.append(",")   # reemplaza ; dentro de textos
        else:
            resultado.append(c)

    return "".join(resultado)


# leer archivo
with open('/content/databases/classicmodels_mysql.sql','r',encoding='utf-8') as f:
    sql = f.read()

# 1️⃣ arreglar comillas escapadas
sql = sql.replace("\\'", "''")

# 2️⃣ eliminar comandos MySQL
sql = re.sub(r'LOCK TABLES.*?;', '', sql, flags=re.I)
sql = re.sub(r'UNLOCK TABLES.*?;', '', sql, flags=re.I)
sql = re.sub(r'SET .*?;', '', sql, flags=re.I)

# 3️⃣ quitar ENGINE
sql = re.sub(r'ENGINE=\w+.*?;', ';', sql)

# 4️⃣ arreglar ; dentro de textos
sql = limpiar_puntoycoma_en_strings(sql)

# guardar SQL limpio
with open('/content/databases/classicmodels_sqlite.sql','w',encoding='utf-8') as f:
    f.write(sql)

print("✅ SQL corregido guardado")

✅ SQL corregido guardado


In [ ]:

with open('/content/databases/classicmodels_sqlite.sql', 'r', encoding='utf-8') as f:
    sql_content = f.read()

conn_mysql = sqlite3.connect(':memory:')

# ── Extraer y crear tablas ─────────────────────────────────────
crear_tablas = re.findall(
    r'CREATE TABLE\s+`?(\w+)`?\s*\((.*?)\)\s*(?:ENGINE|DEFAULT|;)',
    sql_content, flags=re.DOTALL | re.IGNORECASE
)

def limpiar_linea(linea):
    linea = linea.strip().rstrip(',')
    if re.match(r'(PRIMARY KEY|KEY|UNIQUE|CONSTRAINT|INDEX)', linea, re.IGNORECASE):
        return None
    if not linea:
        return None
    linea = re.sub(r'`', '"', linea)
    linea = re.sub(r'\bvarchar\s*\(\s*\d+\s*\)', 'TEXT', linea, flags=re.IGNORECASE)
    linea = re.sub(r'\bchar\s*\(\s*\d+\s*\)', 'TEXT', linea, flags=re.IGNORECASE)
    linea = re.sub(r'\bint\s*\(\s*\d+\s*\)', 'INTEGER', linea, flags=re.IGNORECASE)
    linea = re.sub(r'\bsmallint\s*\(\s*\d+\s*\)', 'INTEGER', linea, flags=re.IGNORECASE)
    linea = re.sub(r'\btinyint\s*\(\s*\d+\s*\)', 'INTEGER', linea, flags=re.IGNORECASE)
    linea = re.sub(r'\bdecimal\s*\(\s*[\d,]+\s*\)', 'REAL', linea, flags=re.IGNORECASE)
    linea = re.sub(r'\bdouble\s*\(\s*[\d,]+\s*\)', 'REAL', linea, flags=re.IGNORECASE)
    linea = re.sub(r'\bfloat\s*\(\s*[\d,]+\s*\)', 'REAL', linea, flags=re.IGNORECASE)
    linea = re.sub(r'\bint\b', 'INTEGER', linea, flags=re.IGNORECASE)
    linea = re.sub(r'\bsmallint\b', 'INTEGER', linea, flags=re.IGNORECASE)
    linea = re.sub(r'\bdate\b', 'TEXT', linea, flags=re.IGNORECASE)
    linea = re.sub(r'\bdatetime\b', 'TEXT', linea, flags=re.IGNORECASE)
    linea = re.sub(r'\bmediumtext\b', 'TEXT', linea, flags=re.IGNORECASE)
    linea = re.sub(r'\blongtext\b', 'TEXT', linea, flags=re.IGNORECASE)
    linea = re.sub(r'\bmediumblob\b', 'BLOB', linea, flags=re.IGNORECASE)
    linea = re.sub(r'\bunsigned\b', '', linea, flags=re.IGNORECASE)
    linea = re.sub(r'\bAUTO_INCREMENT\b', '', linea, flags=re.IGNORECASE)

    linea = linea.strip()
    return linea if linea else None

def convertir_create(nombre, cuerpo):
    columnas_validas = [limpiar_linea(l) for l in cuerpo.split('\n')]
    columnas_validas = [c for c in columnas_validas if c]
    if not columnas_validas:
        return None
    return (f'CREATE TABLE IF NOT EXISTS "{nombre}" (\n  ' +
            ',\n  '.join(columnas_validas) + '\n)')

print("Creando tablas...")
for nombre, cuerpo in crear_tablas:
    sql_create = convertir_create(nombre, cuerpo)
    if not sql_create:
        print(f"  ⚠️ {nombre}: vacío")
        continue
    try:
        conn_mysql.execute(sql_create)
        print(f"  ✅ {nombre}")
    except Exception as e:
        print(f"  ❌ {nombre}: {e}")

conn_mysql.commit()

# ── Parser seguro de INSERTs respetando strings ────────────────
def extraer_inserts_seguro(sql):
    inserts = []
    i = 0
    n = len(sql)
    while i < n:
        match = re.search(r'INSERT INTO', sql[i:], re.IGNORECASE)
        if not match:
            break
        start = i + match.start()
        i = start
        actual = []
        en_string = False
        while i < n:
            c = sql[i]
            if not en_string and c == "'":
                en_string = True
                actual.append(c)
            elif en_string and c == "'":
                if i + 1 < n and sql[i+1] == "'":
                    actual.append("''")
                    i += 2
                    continue
                else:
                    en_string = False
                    actual.append(c)
            elif not en_string and c == ';':
                actual.append(c)
                inserts.append(''.join(actual).strip())
                i += 1
                break
            else:
                actual.append(c)
            i += 1
    return inserts

print("\nInsertando datos...")
inserts_seguros = extraer_inserts_seguro(sql_content)
print(f"INSERT INTO detectados: {len(inserts_seguros)}")

errores = 0
for ins in inserts_seguros:
    ins_clean = ins.replace('`', '"')
    ins_clean = re.sub(r"\\'", "''", ins_clean)
    try:
      conn_mysql.executescript(ins_clean)
        #conn_mysql.execute(ins_clean)
    except Exception as e:
        errores += 1
        print(f"  ❌ ERROR: {e}")
        print(f"     SQL: {ins_clean[:150]}")

conn_mysql.commit()
print(f"Errores: {errores}")

# ── Verificar resultado ────────────────────────────────────────
tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn_mysql)
print("\nFilas por tabla:")
for t in tablas['name']:
    n = pd.read_sql(f'SELECT COUNT(*) as total FROM "{t}"', conn_mysql)['total'][0]
    print(f"  {t}: {n} filas")

Creando tablas...
  ✅ customers
  ✅ employees
  ✅ offices
  ✅ orderdetails
  ✅ orders
  ✅ payments
  ✅ productlines
  ✅ products

Insertando datos...
INSERT INTO detectados: 8
Errores: 0

Filas por tabla:
  customers: 122 filas
  employees: 23 filas
  offices: 7 filas
  orderdetails: 2996 filas
  orders: 326 filas
  payments: 273 filas
  productlines: 7 filas
  products: 74 filas


In [ ]:
def convertir_copy_a_insert(sql):
    lineas = sql.split('\n')
    resultado = []
    en_copy = False
    tabla_actual = None
    columnas_actuales = None
    valores_batch = []

    def limpiar_puntoycoma_en_strings(texto):
        """Reemplaza ; dentro de strings por ,"""
        resultado = []
        en_string = False
        for c in texto:
            if c == "'":
                en_string = not en_string
                resultado.append(c)
            elif c == ";" and en_string:
                resultado.append(",")
            else:
                resultado.append(c)
        return "".join(resultado)

    for linea in lineas:
        linea_strip = linea.strip()

        # Detectar inicio de COPY
        match = re.match(
            r'COPY\s+(?:public\.)?(\w+)\s*\(([^)]+)\)\s+FROM\s+stdin',
            linea_strip, re.IGNORECASE
        )
        if match:
            en_copy = True
            tabla_actual = match.group(1)
            columnas_actuales = match.group(2)
            valores_batch = []
            print(f"  → Detectado COPY para tabla: {tabla_actual}")
            continue

        # Detectar fin de bloque COPY
        if en_copy and linea_strip == '\\.':
            if valores_batch:
                insert = f"INSERT INTO {tabla_actual} ({columnas_actuales}) VALUES\n"
                insert += ',\n'.join(valores_batch) + ';'
                # ── Aplicar limpieza de ; dentro de strings ──
                insert = limpiar_puntoycoma_en_strings(insert)
                resultado.append(insert)
                print(f"  ✅ {tabla_actual}: {len(valores_batch)} filas convertidas")
            en_copy = False
            tabla_actual = None
            columnas_actuales = None
            valores_batch = []
            continue

        # Procesar filas de datos dentro del COPY
        if en_copy and linea_strip:
            campos = linea.split('\t')
            campos_formateados = []
            for campo in campos:
                campo = campo.strip()
                if campo in ('\\N', 'NULL', ''):
                    campos_formateados.append('NULL')
                else:
                    campo = campo.replace("'", "''")
                    campos_formateados.append(f"'{campo}'")
            valores_batch.append(f"  ({', '.join(campos_formateados)})")
            continue

        # Líneas normales fuera de COPY
        if not en_copy:
            resultado.append(linea)

    return '\n'.join(resultado)

# Aplicar conversión
with open('/content/databases/customerservice.sql', 'r', encoding='utf-8') as f:
    sql_pg = f.read()

print("Convirtiendo COPY → INSERT INTO...")
sql_convertido = convertir_copy_a_insert(sql_pg)

with open('/content/databases/customerservice_converted.sql', 'w', encoding='utf-8') as f:
    f.write(sql_convertido)

print(f"\n✅ Total INSERT INTO generados: {sql_convertido.count('INSERT INTO')}")

Convirtiendo COPY → INSERT INTO...
  → Detectado COPY para tabla: cs_customer_calls
  ✅ cs_customer_calls: 108 filas convertidas
  → Detectado COPY para tabla: cs_customer_products
  ✅ cs_customer_products: 101 filas convertidas
  → Detectado COPY para tabla: cs_customers
  ✅ cs_customers: 122 filas convertidas
  → Detectado COPY para tabla: cs_employees
  ✅ cs_employees: 30 filas convertidas
  → Detectado COPY para tabla: cs_products
  ✅ cs_products: 110 filas convertidas

✅ Total INSERT INTO generados: 5


In [ ]:
with open('/content/databases/customerservice_converted.sql', 'r', encoding='utf-8') as f:
    sql_pg = f.read()

conn_pg = sqlite3.connect(':memory:')

def limpiar_sql_postgres(sql):
    sql = re.sub(r'--.*?\n', '\n', sql)
    sql = re.sub(r'\/\*.*?\*\/', '', sql, flags=re.DOTALL)
    sql = re.sub(r'SET .*?;', '', sql)
    sql = re.sub(r'SELECT pg_catalog.*?;', '', sql, flags=re.DOTALL)
    sql = re.sub(r'ALTER TABLE.*?;', '', sql, flags=re.DOTALL)
    sql = re.sub(r'CREATE INDEX.*?;', '', sql, flags=re.DOTALL)
    sql = re.sub(r'REVOKE.*?;', '', sql, flags=re.DOTALL)
    sql = re.sub(r'GRANT.*?;', '', sql, flags=re.DOTALL)
    sql = re.sub(r'character varying\((\d+)\)', 'TEXT', sql, flags=re.IGNORECASE)
    sql = re.sub(r'character varying', 'TEXT', sql, flags=re.IGNORECASE)
    sql = re.sub(r'numeric\([\d,]+\)', 'REAL', sql, flags=re.IGNORECASE)
    sql = re.sub(r'timestamp\s*\(\d+\)\s*without time zone', 'TEXT', sql, flags=re.IGNORECASE)
    sql = re.sub(r'timestamp without time zone', 'TEXT', sql, flags=re.IGNORECASE)
    sql = re.sub(r'integer', 'INTEGER', sql, flags=re.IGNORECASE)
    sql = re.sub(r'public\.', '', sql)
    sql = re.sub(r'DEFAULT nextval.*?\)', '', sql)
    sql = re.sub(r'WITH \(.*?\)', '', sql, flags=re.DOTALL)
    sql = re.sub(r'TABLESPACE.*?;', ';', sql)
    sql = re.sub(r'ON CONFLICT.*?;', ';', sql)
    return sql

sql_pg_limpio = limpiar_sql_postgres(sql_pg)
sentencias_pg = [s.strip() for s in sql_pg_limpio.split(';') if s.strip()]

# ── Ejecutar con detalle de errores ───────────────────────────
errores_pg = 0
for i, sent in enumerate(sentencias_pg):
    try:
        conn_pg.execute(sent)
    except Exception as e:
        errores_pg += 1
        print(f"❌ Error #{i}: {str(e)}")
        print(f"   SQL: {sent[:300]}")
        print("---")

conn_pg.commit()
print(f"\n✅ PostgreSQL cargado — Errores: {errores_pg}")

# ── Tablas y filas ─────────────────────────────────────────────
tablas_pg = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn_pg)
print("\nFilas por tabla:")
for t in tablas_pg['name']:
    n = pd.read_sql(f'SELECT COUNT(*) as total FROM "{t}"', conn_pg)['total'][0]
    print(f"  {t}: {n} filas")


✅ PostgreSQL cargado — Errores: 0

Filas por tabla:
  cs_customer_calls: 108 filas
  cs_customer_products: 101 filas
  cs_customers: 122 filas
  cs_employees: 30 filas
  cs_products: 110 filas


# Conversión de tablas en dataframes


In [ ]:
# obtener lista de nombres de tablas
lista_tablas = tablas['name'].tolist()

# crear diccionario de dataframes
dfs = {t: pd.read_sql(f"SELECT * FROM {t}", conn_mysql) for t in lista_tablas}

# mostrar nombres
print("DataFrames creados:")
print(list(dfs.keys()))

DataFrames creados:
['customers', 'employees', 'offices', 'orderdetails', 'orders', 'payments', 'productlines', 'products']


In [ ]:
df_customers = dfs["customers"]
df_employees = dfs["employees"]
df_offices = dfs["offices"]
df_orderdetails = dfs["orderdetails"]
df_orders = dfs["orders"]
df_payments = dfs["payments"]
df_productlines =dfs["productlines"]
df_products = dfs["products"]

In [ ]:
# obtener tablas de la base PostgreSQL cargada en SQLite
tablas_pg = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn_pg)

print(f"Tablas disponibles:{tablas_pg}")

# convertir nombres de tablas a lista
lista_tablas_pg = tablas_pg['name'].tolist()

# crear dataframes
dfs_pg = {t: pd.read_sql(f"SELECT * FROM {t}", conn_pg) for t in lista_tablas_pg}

# crear variables automáticas
globals().update(dfs_pg)

print("\nDataFrames creados:")
print(list(dfs_pg.keys()))

Tablas disponibles:                   name
0     cs_customer_calls
1  cs_customer_products
2          cs_customers
3          cs_employees
4           cs_products

DataFrames creados:
['cs_customer_calls', 'cs_customer_products', 'cs_customers', 'cs_employees', 'cs_products']


In [ ]:
df_cs_customer_calls = dfs_pg["cs_customer_calls"]
df_cs_customer_products = dfs_pg["cs_customer_products"]
df_cs_customers = dfs_pg["cs_customers"]
df_cs_employees = dfs_pg["cs_employees"]
df_cs_products = dfs_pg["cs_products"]

## Tablas de resumen

In [ ]:
def perfilar_dataframe(df, nombre_tabla):
    resultados = []

    for col in df.columns:
        serie = df[col]
        total = len(serie)
        nulos = serie.isnull().sum()
        pct_nulos = round(nulos / total * 100, 2) if total > 0 else None
        distintos = serie.nunique()

        fila = {
            "Atributo": col,
            "Tipo de Dato": str(serie.dtype),
            "% Nulos": pct_nulos,
            "Cantidad Distintos": distintos,
            "Media": None,
            "Desv. Estándar": None,
            "Moda": None,
            "Mínimo": None,
            "Máximo": None,
            "Valores Distintos (≤5)": None,
            "Patrón String": None
        }

        # Moda (aplica a todos)
        try:
            moda = serie.mode()
            fila["Moda"] = str(moda[0]) if len(moda) > 0 else None
        except:
            fila["Moda"] = None

        # Valores distintos si hay 5 o menos
        if distintos <= 5:
            fila["Valores Distintos (≤5)"] = ", ".join(
                [str(v) for v in serie.dropna().unique()]
            )

        # ── Numéricos ──────────────────────────────────────
        if pd.api.types.is_numeric_dtype(serie):
            fila["Media"]         = round(serie.mean(), 2) if not serie.dropna().empty else None
            fila["Desv. Estándar"] = round(serie.std(), 2) if not serie.dropna().empty else None
            fila["Mínimo"]        = serie.min()
            fila["Máximo"]        = serie.max()

        # ── Fechas ─────────────────────────────────────────
        elif any(k in col.lower() for k in ['date', 'fecha', 'time']):
            try:
                fecha = pd.to_datetime(serie.dropna())
                fila["Mínimo"] = str(fecha.min().date())
                fila["Máximo"] = str(fecha.max().date())
                fila["Media"]  = str(fecha.mean().date())
            except:
                pass

        # ── Strings / Texto ────────────────────────────────
        elif serie.dtype == object:
            muestra = serie.dropna()
            if not muestra.empty:
                # Detectar patrones
                patrones = []

                if muestra.str.match(r'^\d+$').all():
                    patrones.append("Solo numérico")
                elif muestra.str.match(r'^[a-zA-Z\s]+$').all():
                    patrones.append("Solo alfabético")
                elif muestra.str.match(r'^[a-zA-Z0-9\s]+$').all():
                    patrones.append("Alfanumérico")

                if muestra.str.contains(r'@').any():
                    patrones.append("Contiene email (@)")

                if muestra.str.match(r'^\+?[\d\s\-().]+$').all():
                    patrones.append("Formato teléfono")

                if muestra.str.match(r'^\d{4}-\d{2}-\d{2}').all():
                    patrones.append("Formato fecha YYYY-MM-DD")

                if muestra.str.match(r'^[A-Z]{1,2}\d+').all():
                    patrones.append("Código (letras+números)")

                if muestra.str.match(r'^\d{4,10}$').all():
                    patrones.append("Código postal / ID numérico")

                longitudes = muestra.str.len()
                if longitudes.nunique() == 1:
                    patrones.append(f"Longitud fija: {int(longitudes.iloc[0])}")

                fila["Patrón String"] = " | ".join(patrones) if patrones else "Texto libre"

        resultados.append(fila)

    df_perfil = pd.DataFrame(resultados)
    print(f"✅ Perfilamiento de '{nombre_tabla}' — {len(df.columns)} columnas, {len(df)} filas")
    return df_perfil

### Tablas resumen fuente de datos: classicmodels_mysql

---



In [ ]:
perfilar_dataframe(df_customers, "customers")


✅ Perfilamiento de 'customers' — 13 columnas, 122 filas


,Atributo,Tipo de Dato,% Nulos,Cantidad Distintos,Media,Desv. Estándar,Moda,Mínimo,Máximo,Valores Distintos (≤5),Patrón String
0,customerNumber,int64,0.00,122,296.40,117.08,103,103.0,496.0,None,None
1,customerName,object,0.00,122,NaN,NaN,ANG Resellers,NaN,NaN,None,Texto libre
2,contactLastName,object,0.00,108,NaN,NaN,Young,NaN,NaN,None,Texto libre
3,contactFirstName,object,0.00,111,NaN,NaN,Julie,NaN,NaN,None,Texto libre
4,phone,object,0.00,121,NaN,NaN,6175558555,NaN,NaN,None,Texto libre
5,addressLine1,object,0.00,122,NaN,NaN,1 rue Alsace-Lorraine,NaN,NaN,None,Texto libre
6,addressLine2,object,81.97,21,NaN,NaN,Suite 101,NaN,NaN,None,Texto libre
7,city,object,0.00,96,NaN,NaN,Madrid,NaN,NaN,None,Texto libre
8,state,object,59.84,18,NaN,NaN,CA,NaN,NaN,None,Texto libre
9,postalCode,object,5.74,94,NaN,NaN,10022,NaN,NaN,None,Texto libre


In [ ]:
perfilar_dataframe(df_employees, "employees")

✅ Perfilamiento de 'employees' — 8 columnas, 23 filas


,Atributo,Tipo de Dato,% Nulos,Cantidad Distintos,Media,Desv. Estándar,Moda,Mínimo,Máximo,Valores Distintos (≤5),Patrón String
0,employeeNumber,int64,0.00,23,1335.39,223.57,1002,1002.0,1702.0,None,None
1,lastName,object,0.00,19,NaN,NaN,Patterson,NaN,NaN,None,Solo alfabético
2,firstName,object,0.00,21,NaN,NaN,Gerard,NaN,NaN,None,Solo alfabético
3,extension,object,0.00,20,NaN,NaN,x102,NaN,NaN,None,Alfanumérico
4,email,object,0.00,22,NaN,NaN,jfirrelli@classicmodelcars.com,NaN,NaN,None,Contiene email (@)
5,officeCode,object,0.00,7,NaN,NaN,1,NaN,NaN,None,Solo numérico | Formato teléfono | Longitud fi...
6,reportsTo,float64,4.35,6,1117.41,120.17,1102.0,1002.0,1621.0,None,None
7,jobTitle,object,0.00,7,NaN,NaN,Sales Rep,NaN,NaN,None,Texto libre


In [ ]:
perfil_employees = perfilar_dataframe(df_employees, "employees")
perfil_employees

In [ ]:
perfilar_dataframe(df_offices, "offices")

✅ Perfilamiento de 'offices' — 9 columnas, 7 filas


,Atributo,Tipo de Dato,% Nulos,Cantidad Distintos,Media,Desv. Estándar,Moda,Mínimo,Máximo,Valores Distintos (≤5),Patrón String
0,officeCode,object,0.00,7,None,None,1,None,None,None,Solo numérico | Formato teléfono | Longitud fi...
1,city,object,0.00,7,None,None,Boston,None,None,None,Solo alfabético
2,phone,object,0.00,7,None,None,+1 212 555 3000,None,None,None,Formato teléfono
3,addressLine1,object,0.00,7,None,None,100 Market Street,None,None,None,Texto libre
4,addressLine2,object,28.57,5,None,None,Floor #2,None,None,"Suite 300, Suite 102, apt. 5A, Floor #2, Level 7",Texto libre
5,state,object,42.86,4,None,None,CA,None,None,"CA, MA, NY, Chiyoda-Ku",Texto libre
6,country,object,0.00,5,None,None,USA,None,None,"USA, France, Japan, Australia, UK",Solo alfabético
7,postalCode,object,0.00,7,None,None,02107,None,None,None,Texto libre
8,territory,object,0.00,4,None,None,NA,None,None,"NA, EMEA, Japan, APAC",Solo alfabético


In [ ]:
perfilar_dataframe(df_orderdetails, "orderdetails")

✅ Perfilamiento de 'orderdetails' — 5 columnas, 2996 filas


,Atributo,Tipo de Dato,% Nulos,Cantidad Distintos,Media,Desv. Estándar,Moda,Mínimo,Máximo,Valores Distintos (≤5),Patrón String
0,orderNumber,int64,0.0,326,10260.35,92.48,10106,10100.00,10425.0,None,None
1,productCode,object,0.0,109,NaN,NaN,S18_3232,NaN,NaN,None,Código (letras+números)
2,quantityOrdered,int64,0.0,61,35.22,9.83,34,6.00,97.0,None,None
3,priceEach,float64,0.0,1573,90.77,36.58,98.48,26.55,214.3,None,None
4,orderLineNumber,int64,0.0,18,6.43,4.20,1,1.00,18.0,None,None


In [ ]:
perfilar_dataframe(df_orders, "orders")

✅ Perfilamiento de 'orders' — 7 columnas, 326 filas


,Atributo,Tipo de Dato,% Nulos,Cantidad Distintos,Media,Desv. Estándar,Moda,Mínimo,Máximo,Valores Distintos (≤5),Patrón String
0,orderNumber,int64,0.00,326,10262.5,94.25,10100,10100,10425,None,None
1,orderDate,object,0.00,265,2004-05-18,NaN,2004-11-24,2003-01-06,2005-05-31,None,None
2,requiredDate,object,0.00,264,2004-05-27,NaN,2004-12-01,2003-01-13,2005-06-11,None,None
3,shippedDate,object,4.29,247,2004-05-12,NaN,2004-09-14,2003-01-10,2005-05-20,None,None
4,status,object,0.00,6,None,NaN,Shipped,None,None,None,Solo alfabético
5,comments,object,75.46,37,None,NaN,They want to reevaluate their terms agreement ...,None,None,None,Texto libre
6,customerNumber,int64,0.00,98,263.83,121.33,141,103,496,None,None


In [ ]:
perfilar_dataframe(df_payments, "payments")

✅ Perfilamiento de 'payments' — 4 columnas, 273 filas


,Atributo,Tipo de Dato,% Nulos,Cantidad Distintos,Media,Desv. Estándar,Moda,Mínimo,Máximo,Valores Distintos (≤5),Patrón String
0,customerNumber,int64,0.0,98,271.19,120.07,141,103,496,None,None
1,checkNumber,object,0.0,273,None,NaN,AB661578,None,None,None,Alfanumérico
2,paymentDate,object,0.0,232,2004-05-05,NaN,2003-11-18,2003-01-16,2005-06-09,None,None
3,amount,float64,0.0,273,32431.65,20997.12,615.45,615.45,120166.58,None,None


In [ ]:
perfilar_dataframe(df_products, "products")

✅ Perfilamiento de 'products' — 9 columnas, 74 filas


,Atributo,Tipo de Dato,% Nulos,Cantidad Distintos,Media,Desv. Estándar,Moda,Mínimo,Máximo,Valores Distintos (≤5),Patrón String
0,productCode,object,0.0,74,NaN,NaN,S10_1678,NaN,NaN,None,Código (letras+números)
1,productName,object,0.0,74,NaN,NaN,18th Century Vintage Horse Carriage,NaN,NaN,None,Texto libre
2,productLine,object,0.0,7,NaN,NaN,Classic Cars,NaN,NaN,None,Solo alfabético
3,productScale,object,0.0,8,NaN,NaN,1:18,NaN,NaN,None,Texto libre
4,productVendor,object,0.0,13,NaN,NaN,Motor City Art Classics,NaN,NaN,None,Alfanumérico
5,productDescription,object,0.0,65,NaN,NaN,"Turnable front wheels, steering function, deta...",NaN,NaN,None,Texto libre
6,quantityInStock,int64,0.0,74,5000.57,3012.55,68,68.00,9997.00,None,None
7,buyPrice,float64,0.0,72,56.85,22.71,33.3,21.75,103.42,None,None
8,MSRP,float64,0.0,74,106.44,41.57,40.23,40.23,214.30,None,None


In [ ]:
perfilar_dataframe(df_productlines, "productlines")

✅ Perfilamiento de 'productlines' — 4 columnas, 7 filas


,Atributo,Tipo de Dato,% Nulos,Cantidad Distintos,Media,Desv. Estándar,Moda,Mínimo,Máximo,Valores Distintos (≤5),Patrón String
0,productLine,object,0.0,7,None,None,Classic Cars,None,None,None,Solo alfabético
1,textDescription,object,0.0,7,None,None,Attention car enthusiasts: Make your wildest c...,None,None,None,Texto libre
2,htmlDescription,object,100.0,0,None,None,None,None,None,,None
3,image,object,100.0,0,None,None,None,None,None,,None


### Tablas resumen fuente de datos: customservice

In [ ]:
perfilar_dataframe(df_cs_customer_calls, "cs_customer_calls")

✅ Perfilamiento de 'cs_customer_calls' — 5 columnas, 108 filas


,Atributo,Tipo de Dato,% Nulos,Cantidad Distintos,Media,Desv. Estándar,Moda,Mínimo,Máximo,Valores Distintos (≤5),Patrón String
0,employeenumber,int64,0.0,30,16.57,8.45,18,1,30,None,None
1,customernumber,int64,0.0,54,300.4,117.19,276,112,496,None,None
2,productcode,object,0.0,47,None,NaN,S700_1691,None,None,None,Código (letras+números)
3,text,object,0.0,108,None,NaN,Aenean a sapien suscipit rhoncus odio nec cond...,None,None,None,Texto libre
4,date,object,0.0,100,2004-03-19,NaN,2003-04-11 00:00:00,2003-02-19,2005-05-09,None,None


In [ ]:
perfilar_dataframe(df_cs_customer_products, "cs_customer_products")

✅ Perfilamiento de 'cs_customer_products' — 2 columnas, 101 filas


,Atributo,Tipo de Dato,% Nulos,Cantidad Distintos,Media,Desv. Estándar,Moda,Mínimo,Máximo,Valores Distintos (≤5),Patrón String
0,customernumber,int64,0.0,75,299.91,114.64,172,112.0,496.0,None,None
1,productcode,object,0.0,64,NaN,NaN,S50_1514,NaN,NaN,None,Código (letras+números)


In [ ]:
perfilar_dataframe(df_cs_customers, "cs_customers")

✅ Perfilamiento de 'cs_customers' — 10 columnas, 122 filas


,Atributo,Tipo de Dato,% Nulos,Cantidad Distintos,Media,Desv. Estándar,Moda,Mínimo,Máximo,Valores Distintos (≤5),Patrón String
0,customernumber,int64,0.00,122,296.4,117.08,103,103.0,496.0,None,None
1,contactlastname,object,0.00,108,NaN,NaN,Young,NaN,NaN,None,Texto libre
2,contactfirstname,object,0.00,105,NaN,NaN,Julie,NaN,NaN,None,Texto libre
3,phone,object,0.00,121,NaN,NaN,6175558555,NaN,NaN,None,Formato teléfono
4,addressline1,object,0.00,122,NaN,NaN,1 rue Alsace-Lorraine,NaN,NaN,None,Texto libre
5,addressline2,object,81.97,21,NaN,NaN,Suite 101,NaN,NaN,None,Texto libre
6,city,object,0.00,95,NaN,NaN,Madrid,NaN,NaN,None,Texto libre
7,state,object,59.84,18,NaN,NaN,CA,NaN,NaN,None,Texto libre
8,postalcode,object,5.74,94,NaN,NaN,10022,NaN,NaN,None,Texto libre
9,country,object,0.00,27,NaN,NaN,USA,NaN,NaN,None,Solo alfabético


In [ ]:
perfilar_dataframe(df_cs_employees, "cs_employees")

✅ Perfilamiento de 'cs_employees' — 4 columnas, 30 filas


,Atributo,Tipo de Dato,% Nulos,Cantidad Distintos,Media,Desv. Estándar,Moda,Mínimo,Máximo,Valores Distintos (≤5),Patrón String
0,employeenumber,int64,0.0,30,15.5,8.8,1,1.0,30.0,None,None
1,lastname,object,0.0,30,NaN,NaN,Appell,NaN,NaN,None,Solo alfabético
2,firstname,object,0.0,30,NaN,NaN,Angelique,NaN,NaN,None,Solo alfabético
3,email,object,0.0,30,NaN,NaN,appell@classicmodelcars.com,NaN,NaN,None,Contiene email (@)


In [ ]:
perfilar_dataframe(df_cs_products, "cs_products")

✅ Perfilamiento de 'cs_products' — 5 columnas, 110 filas


,Atributo,Tipo de Dato,% Nulos,Cantidad Distintos,Media,Desv. Estándar,Moda,Mínimo,Máximo,Valores Distintos (≤5),Patrón String
0,productcode,object,0.0,110,None,None,S10_1678,None,None,None,Código (letras+números)
1,productname,object,0.0,110,None,None,18th Century Vintage Horse Carriage,None,None,None,Texto libre
2,productscale,object,0.0,8,None,None,1:18,None,None,None,Texto libre
3,productvendor,object,0.0,13,None,None,Classic Metal Creations,None,None,None,Alfanumérico
4,productdescription,object,0.0,95,None,None,"Turnable front wheels, steering function, deta...",None,None,None,Texto libre


## Análisis fuentes comunes

Las dos fuentes de datos tienen sus entidades principales en común: clientes, empleados y productos. En esta sección vamos a ver si entre las fuentes hay registros en común.


In [ ]:
def comparar_ids(serie1, nombre1, serie2, nombre2):
    ids1 = serie1.drop_duplicates().sort_values().reset_index(drop=True)
    ids2 = serie2.drop_duplicates().sort_values().reset_index(drop=True)

    print(f"{nombre1}: {len(ids1)} registros")
    print(f"{nombre2}: {len(ids2)} registros")

    if len(ids1) == len(ids2):
        resultado = (ids1.values == ids2.values).all()
        print(f"\n¿Todos los IDs coinciden? {resultado}")
    else:
        print("\n⚠️ Las series tienen diferente tamaño")

        solo_1 = set(ids1) - set(ids2)
        solo_2 = set(ids2) - set(ids1)
        en_comun = set(ids1) & set(ids2)

        print(f"En común:       {len(en_comun)}")
        print(f"Solo en {nombre1}: {len(solo_1)} → {sorted(solo_1)}")
        print(f"Solo en {nombre2}: {len(solo_2)} → {sorted(solo_2)}")


### Clientes

In [ ]:
df_customers.head()

,customerNumber,customerName,contactLastName,contactFirstName,phone,addressLine1,addressLine2,city,state,postalCode,country,salesRepEmployeeNumber,creditLimit
0,103,Atelier graphique,Schmitt,Carine,40.32.2555,"54, rue Royale",None,Nantes,None,44000,France,1370.0,21000.0
1,112,Signal Gift Stores,King,Jean,7025551838,8489 Strong St.,None,Las Vegas,NV,83030,USA,1166.0,71800.0
2,114,"Australian Collectors, Co.",Ferguson,Peter,03 9520 4555,636 St Kilda Road,Level 3,Melbourne,Victoria,3004,Australia,1611.0,117300.0
3,119,La Rochelle Gifts,Labrune,Janine,40.67.8555,"67, rue des Cinquante Otages",None,Nantes,None,44000,France,1370.0,118200.0
4,121,Baane Mini Imports,Bergulfsen,Jonas,07-98 9555,Erling Skakkes gate 78,None,Stavern,None,4110,Norway,1504.0,81700.0


In [ ]:
df_cs_customers.head()

,customernumber,contactlastname,contactfirstname,phone,addressline1,addressline2,city,state,postalcode,country
0,103,Schmitt,Carine,40.32.2555,"54, rue Royale",None,Nantes,None,44000,France
1,112,King,Jean,7025551838,8489 Strong St.,None,Las Vegas,NV,83030,USA
2,114,Ferguson,Peter,03 9520 4555,636 St Kilda Road,Level 3,Melbourne,Victoria,3004,Australia
3,119,Labrune,Janine,40.67.8555,"67, rue des Cinquante Otages",None,Nantes,None,44000,France
4,121,Bergulfsen,Jonas,07-98 9555,Erling Skakkes gate 78,None,Stavern,None,4110,Norway


Se toma como id la columna `customer_number` para realizar la comparación entre ambas fuentes de datos


In [ ]:
comparar_ids(
    df_customers["customerNumber"], "classicmodels",
    df_cs_customers["customernumber"], "customerservice"
)

classicmodels: 122 registros
customerservice: 122 registros

¿Todos los IDs coinciden? True


### Empleados

In [ ]:
df_employees.head()

,employeeNumber,lastName,firstName,extension,email,officeCode,reportsTo,jobTitle
0,1002,Murphy,Diane,x5800,dmurphy@classicmodelcars.com,1,NaN,President
1,1056,Patterson,Mary,x4611,mpatterso@classicmodelcars.com,1,1002.0,VP Sales
2,1076,Firrelli,Jeff,x9273,jfirrelli@classicmodelcars.com,1,1002.0,VP Marketing
3,1088,Patterson,William,x4871,wpatterson@classicmodelcars.com,6,1056.0,Sales Manager (APAC)
4,1102,Bondur,Gerard,x5408,gbondur@classicmodelcars.com,4,1056.0,Sale Manager (EMEA)


In [ ]:
df_cs_employees.head()

,employeenumber,lastname,firstname,email
0,1,Lucas,Diego,lucas@classicmodelcars.com
1,2,Stockard,Fleta,stockard@classicmodelcars.com
2,3,Lindstedt,Margery,lindstedt@classicmodelcars.com
3,4,Nissen,Stacy,nissen@classicmodelcars.com
4,5,Gorecki,Camilla,gorecki@classicmodelcars.com


In [ ]:
comparar_ids(
    df_employees["email"], "classicmodels",
    df_cs_employees["email"], "customerservice"
)

classicmodels: 22 registros
customerservice: 30 registros

⚠️ Las series tienen diferente tamaño
En común:       0
Solo en classicmodels: 22 → ['abow@classicmodelcars.com', 'afixter@classicmodelcars.com', 'bjones@classicmodelcars.com', 'dmurphy@classicmodelcars.com', 'ftseng@classicmodelcars.com', 'gbondur@classicmodelcars.com', 'ghernande@classicmodelcars.com', 'gvanauf@classicmodelcars.com', 'jfirrelli@classicmodelcars.com', 'lbondur@classicmodelcars.com', 'lbott@classicmodelcars.com', 'ljennings@classicmodelcars.com', 'lthompson@classicmodelcars.com', 'mgerard@classicmodelcars.com', 'mnishi@classicmodelcars.com', 'mpatterso@classicmodelcars.com', 'pcastillo@classicmodelcars.com', 'pmarsh@classicmodelcars.com', 'spatterson@classicmodelcars.com', 'tking@classicmodelcars.com', 'wpatterson@classicmodelcars.com', 'ykato@classicmodelcars.com']
Solo en customerservice: 30 → ['appell@classicmodelcars.com', 'barak@classicmodelcars.com', 'caul@classicmodelcars.com', 'dall@classicmodelcars.com

### Productos


In [ ]:
df_products.head()

,productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP
0,S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,1:10,Min Lin Diecast,"This replica features working kickstand, front...",7933,48.81,95.70
1,S10_1949,1952 Alpine Renault 1300,Classic Cars,1:10,Classic Metal Creations,"Turnable front wheels, steering function, deta...",7305,98.58,214.30
2,S10_2016,1996 Moto Guzzi 1100i,Motorcycles,1:10,Highway 66 Mini Classics,"Official Moto Guzzi logos and insignias, saddl...",6625,68.99,118.94
3,S10_4698,2003 Harley-Davidson Eagle Drag Bike,Motorcycles,1:10,Red Start Diecast,"Model features, official Harley Davidson logos...",5582,91.02,193.66
4,S10_4757,1972 Alfa Romeo GTA,Classic Cars,1:10,Motor City Art Classics,"Features include: Turnable front wheels, steer...",3252,85.68,136.00


In [ ]:
df_cs_products.head()

,productcode,productname,productscale,productvendor,productdescription
0,S24_2011,18th century schooner,1:24,Carousel DieCast Legends,All wood with canvas sails. Many extras includ...
1,S24_2022,1938 Cadillac V-16 Presidential Limousine,1:24,Classic Metal Creations,This 1:24 scale precision die cast replica of ...
2,S24_2300,1962 Volkswagen Microbus,1:24,Autoart Studio Design,This 1:18 scale die cast replica of the 1962 M...
3,S24_2360,1982 Ducati 900 Monster,1:24,Highway 66 Mini Classics,"Features two-tone paint with chrome accents, s..."
4,S24_2766,1949 Jaguar XK 120,1:24,Classic Metal Creations,Precision-engineered from original Jaguar spec...


In [ ]:
comparar_ids(
    df_products["productCode"], "classicmodels",
    df_cs_products["productcode"], "customerservice"
)

classicmodels: 74 registros
customerservice: 110 registros

⚠️ Las series tienen diferente tamaño
En común:       74
Solo en classicmodels: 0 → []
Solo en customerservice: 36 → ['S18_3278', 'S18_3320', 'S18_3482', 'S18_3685', 'S18_3782', 'S18_3856', 'S18_4027', 'S18_4409', 'S18_4522', 'S18_4600', 'S18_4668', 'S18_4721', 'S18_4933', 'S24_1046', 'S24_1444', 'S24_1578', 'S24_1628', 'S24_1785', 'S24_1937', 'S24_2000', 'S24_2011', 'S24_2022', 'S24_2300', 'S24_2360', 'S24_2766', 'S24_2840', 'S24_2841', 'S24_2887', 'S24_2972', 'S24_3151', 'S24_3191', 'S24_3371', 'S24_3420', 'S24_3432', 'S24_3816', 'S24_3856']


## Conclusiones

### Clientes
Los 122 registros de la tabla `customers` (classicmodels) y `cs_customers` (customerservice) son **idénticos**: mismo número de registros, mismos identificadores (`customerNumber`) y mismos valores en todos los campos comunes (nombre de contacto, teléfono, dirección, ciudad, estado, código postal y país). Esto indica que ambas fuentes comparten exactamente la misma base de clientes, y que la entidad **Cliente** está replicada en los dos sistemas con consistencia total. No se detectaron duplicados ni discrepancias entre fuentes.

### Empleados
Los empleados de `classicmodels` (22 registros, fuerza de ventas) y los de `customerservice` (30 registros, agentes del call center) son **poblaciones completamente distintas**: no se encontró ningún correo electrónico en común entre las dos fuentes. Esto es coherente con la separación organizacional entre el área comercial y el área de servicio al cliente. Cada sistema gestiona su propio personal de forma independiente.

### Productos
El catálogo de productos de `classicmodels` (74 registros en `products`) está **contenido completamente** dentro de `cs_products` (110 registros). Todos los códigos de producto de classicmodels aparecen en customerservice, pero customerservice tiene 36 productos adicionales que classicmodels no registra. Cabe destacar que en la fase de perfilamiento individual, `cs_products` apareció con 0 filas debido a un problema de carga en SQLite; el análisis cruzado se realizó con los datos correctamente cargados (110 filas).

### Calidad de datos general
- Los campos opcionales como `addressLine2` (~82% nulos) y `state` (~60% nulos en clientes) presentan altos porcentajes de nulos esperados por la naturaleza opcional de esos campos.
- La columna `comments` en `orders` tiene un 75.46% de nulos, lo cual es normal dado que los comentarios son excepcionales.
- Las columnas `htmlDescription` e `image` en `productlines` están al 100% nulas, lo que sugiere que ese contenido no está siendo utilizado o aún no ha sido diligenciado.
- Los datos numéricos y de fechas presentan rangos coherentes y sin anomalías evidentes.
- No se detectaron duplicados en ninguna de las claves primarias de las tablas analizadas.